fichier de création de métadata. Permet d'éviter de recalculer ces trucs à chaque nouveau code. On met au propre les métadonnées sorties des fichiers de comportements MEG, pour leurs rajouter aussi les paramètres fittés. Il faudrait le faire pour tous les paramètres fittés (je crois que c'est pas le cas), pour chaque modèle (pour le moment j'ai fitté que le modèle H1, pas H0bis, H0). Il faudrait aussi le faire sur les données en ligne et ENS, parce que ces codes sont particulièrement longs. 

In [39]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import glob
import os
import cmath 
import json
import seaborn as sns
from collections import Counter
from scipy.stats import linregress
import PyQt5
# %matplotlib qt
import mne
from tqdm import tqdm
from scipy.io import loadmat

In [40]:
SUBJS = ["07", "09", "10", "11", "12", "14"]
SUBJS_NOT_GOOD = ["01", "02", "03", "05", "06", "08"]

# **Fonctions**

In [41]:
def metadata_file_creation (subject) : 
    print("printing subject ", subject, "...")
    metadatas = pd.read_csv(f"/Volumes/My Passport/Data comportement/data clean behavior MEG/FRINT_MEG_S{subject}.csv", index_col = 0)
    
    metadatas.columns = metadatas.columns.str.strip()
    
    
    df_filtered = metadatas.copy()
    # Exemple: sélection des colonnes rotangle_1 à rotangle_7
    rotangle_cols = [f'rotangle_{i}' for i in range(1, 8)]
    rotation_cols = [f'rotations_{i}' for i in range(1, 8)]
    presangle_cols = [f'presangle_{i}' for i in range(1, 8)]
    df_filtered['presented_angles'] = df_filtered[presangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['expected_angles'] = df_filtered[rotangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['stimulus_rotations'] = df_filtered[rotation_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered["expected_angles_wo_rot"] = df_filtered["presavg"]
    df_filtered["length of sequence"] = df_filtered["Seqlen"]
    df_filtered = df_filtered.drop(columns=rotangle_cols + rotation_cols + presangle_cols)
    df_filtered = df_filtered.drop(columns=["optimal", "presavg", "Seqlen"])
    df_filtered.reset_index(drop=True, inplace=True)
    df_filtered = df_filtered.dropna(subset=["final_angle"])
    
    df_radian = df_filtered.copy().drop(columns=["final_angle", "GenMean", "target", "expected_angles_wo_rot"])

    df_radian['final_angle'] = np.deg2rad(df_filtered['final_angle'])
    df_radian["target"] = np.deg2rad(df_filtered['target'])
    df_radian["GenMean"] = np.deg2rad(df_filtered['GenMean'])
    df_radian["expected_angles_wo_rot"] = np.deg2rad(df_filtered['expected_angles_wo_rot'])
    df_radian['stimulus_rotations_rad'] = df_filtered['stimulus_rotations'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['stimulus_rotations_degrees'] = df_filtered['stimulus_rotations']
    df_radian['presented_angles_rad'] = df_filtered['presented_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['expected_angles_rad'] = df_filtered['expected_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))

    df_radian["mean_rotation"] = df_radian["stimulus_rotations_rad"].apply(np.mean) 
    metadatas_clean = df_radian.copy()
    
    if float(subject)%3 == 0 :
        color_mapping = {np.deg2rad(0): 'blue',  np.deg2rad(45): 'green', np.deg2rad(-45): 'red'}
    elif float(subject)%3 == 1 :
        color_mapping = {np.deg2rad(0): 'red',  np.deg2rad(45): 'blue', np.deg2rad(-45): 'green'}
    elif float(subject)%3 == 2 : 
        color_mapping = {np.deg2rad(0): 'green',  np.deg2rad(45): 'red', np.deg2rad(-45): 'blue'}
    
    
    colors = []
    colors_extended = []
    for i in range(len(metadatas_clean)) : 
        colors_one_extended = []
        colors_one = []
        for x in metadatas_clean["stimulus_rotations_rad"].iloc[i]:
            colors_one.append(color_mapping[x])
            colors_one_extended.extend(color_mapping[x])
        colors.append(colors_one)
        colors_extended.extend(colors_one)

    metadatas_clean["color_seen"] = colors
    
    
    data = loadmat(f"/Volumes/My Passport/Data comportement/M5_MEG_FIT/{subject}_M5_VBMC.mat")
    out = data["out_vbmc"][0,0]
    average = out["xavg"].squeeze()
    average_deg = ((np.array(average) *180/np.pi) )  # convert the angles in degrees
    d_rot_p = average_deg[2] 
    d_rot_m = average_deg[3]
    
    
    block_extended = []
    rotation_radians_extended = []
    rotation_degrees_extended = []
    expected_angles_extended = []
    expected_angles_rad_extended = []

    presented_angles_rad_extended = []
    presented_angles_extended = []
    fitted_value = []
    seq_length_extended = []

    count = 0
    sequence_position_extended = []

    for i in range(len(metadatas_clean)) :
        seq_len = len(metadatas_clean["stimulus_rotations"].iloc[i])
        # print(seq_len)
        for pos in range(seq_len) :
            block_extended.append(metadatas_clean["Block"].iloc[i])
            presented_angles_extended.append(
                metadatas_clean["presented_angles"].iloc[i][pos]
            )
            presented_angles_rad_extended.append(
                metadatas_clean["presented_angles_rad"].iloc[i][pos] % np.pi
            )
            rotation_radians_extended.append(
                metadatas_clean["stimulus_rotations"].iloc[i][pos] % np.pi
            )
            rotation_degrees_extended.append(
                metadatas_clean["stimulus_rotations_degrees"].iloc[i][pos]
            )
            expected_angles_extended.append(
                metadatas_clean["expected_angles"].iloc[i][pos]
            )
            expected_angles_rad_extended.append(
                metadatas_clean["expected_angles_rad"].iloc[i][pos] % np.pi
            )
            
            seq_length_extended.append(
                df_radian["length of sequence"].iloc[i]
            )

            sequence_position_extended.append(pos+1)  # <--- ICI
            
            
            
    # fit 
    fitted_expected_angles_extended = []
    fitted_expected_angles_rad_extended = []
    for rotation, presented_angle in zip(rotation_degrees_extended, presented_angles_extended) : 
        if rotation == 45.0 :
            fitted_expected_angles_extended.append(presented_angle + d_rot_p)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle + d_rot_p))
            fitted_value.append(d_rot_p)
        elif rotation == -45.0 :
            fitted_expected_angles_extended.append(presented_angle + d_rot_m)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle + d_rot_m))
            fitted_value.append(d_rot_m)

        elif rotation == 0.0 :
            fitted_expected_angles_extended.append(presented_angle)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle))
            fitted_value.append(0)

    
    metadata_to_csv = pd.DataFrame()
    metadata_to_csv["stim_color"] = colors_extended
    metadata_to_csv["presented_angles_rad"] = presented_angles_rad_extended
    metadata_to_csv["block"] = block_extended
    metadata_to_csv["rotation_in_radian"] = rotation_radians_extended
    metadata_to_csv["rotation_in_degrees"] = rotation_degrees_extended
    metadata_to_csv["fitted_value_deg"] = fitted_value
    metadata_to_csv["expected_angles"] = expected_angles_extended
    metadata_to_csv["expected_angles_rad"] = expected_angles_rad_extended
    metadata_to_csv["presented_angles"] = presented_angles_extended
    metadata_to_csv["fitted_expected_angles"] = fitted_expected_angles_extended
    metadata_to_csv["fitted_expected_angles_rad"] = fitted_expected_angles_rad_extended
    metadata_to_csv["sequence_position"] = sequence_position_extended 
    metadata_to_csv["sequence_length"] = seq_length_extended 
    

    metadata_to_csv.to_csv(f'/Volumes/My Passport/Data comportement/data clean behavior MEG/metadata_for_epochs/FRINT_MEG_S{subject}_cleaned_metadata.csv', index=False)
    print(f"subject {subject} done !")
    return metadata_to_csv 

In [42]:
range(5)

range(0, 5)

In [43]:
def metadata_file_creation_notgood (subject) : 
    print("printing subject ", subject, "...")
    metadatas = pd.read_csv(f"/Volumes/My Passport/Data comportement/data clean behavior MEG/FRINT_MEG_S{subject}.csv", index_col = 0)
    
    metadatas.columns = metadatas.columns.str.strip()
    
    
    df_filtered = metadatas.copy()
    # Exemple: sélection des colonnes rotangle_1 à rotangle_7
    rotangle_cols = [f'rotangle_{i}' for i in range(1, 8)]
    rotation_cols = [f'rotations_{i}' for i in range(1, 8)]
    presangle_cols = [f'presangle_{i}' for i in range(1, 8)]
    df_filtered['presented_angles'] = df_filtered[presangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['expected_angles'] = df_filtered[rotangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['stimulus_rotations'] = df_filtered[rotation_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered["expected_angles_wo_rot"] = df_filtered["presavg"]
    df_filtered["length of sequence"] = df_filtered["Seqlen"]
    df_filtered = df_filtered.drop(columns=rotangle_cols + rotation_cols + presangle_cols)
    df_filtered = df_filtered.drop(columns=["optimal", "presavg", "Seqlen"])
    df_filtered.reset_index(drop=True, inplace=True)
    df_filtered = df_filtered.dropna(subset=["final_angle"])
    
    df_radian = df_filtered.copy().drop(columns=["final_angle", "GenMean", "target", "expected_angles_wo_rot"])

    df_radian['final_angle'] = np.deg2rad(df_filtered['final_angle'])
    df_radian["target"] = np.deg2rad(df_filtered['target'])
    df_radian["GenMean"] = np.deg2rad(df_filtered['GenMean'])
    df_radian["expected_angles_wo_rot"] = np.deg2rad(df_filtered['expected_angles_wo_rot'])
    df_radian['stimulus_rotations_rad'] = df_filtered['stimulus_rotations'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['stimulus_rotations_degrees'] = df_filtered['stimulus_rotations']
    df_radian['presented_angles_rad'] = df_filtered['presented_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['expected_angles_rad'] = df_filtered['expected_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))

    df_radian["mean_rotation"] = df_radian["stimulus_rotations_rad"].apply(np.mean) 
    metadatas_clean = df_radian.copy()
    
    if float(subject)%3 == 0 :
        color_mapping = {np.deg2rad(0): 'blue',  np.deg2rad(45): 'green', np.deg2rad(-45): 'red'}
    elif float(subject)%3 == 1 :
        color_mapping = {np.deg2rad(0): 'red',  np.deg2rad(45): 'blue', np.deg2rad(-45): 'green'}
    elif float(subject)%3 == 2 : 
        color_mapping = {np.deg2rad(0): 'green',  np.deg2rad(45): 'red', np.deg2rad(-45): 'blue'}
    
    
    colors = []
    colors_extended = []
    for i in range(len(metadatas_clean)) : 
        colors_one_extended = []
        colors_one = []
        for x in metadatas_clean["stimulus_rotations_rad"].iloc[i]:
            colors_one.append(color_mapping[x])
            colors_one_extended.extend(color_mapping[x])
        colors.append(colors_one)
        colors_extended.extend(colors_one)

    metadatas_clean["color_seen"] = colors
        
    
    block_extended = []
    rotation_radians_extended = []
    rotation_degrees_extended = []
    expected_angles_extended = []
    expected_angles_rad_extended = []

    presented_angles_rad_extended = []
    presented_angles_extended = []

    count = 0
    for i in range(len(metadatas_clean)) :
        for element in range(len(metadatas_clean["stimulus_rotations"].iloc[i])) :
            block_extended.append(metadatas_clean["Block"].iloc[i])
            presented_angles_extended.append(metadatas_clean["presented_angles"].iloc[i][element])
            presented_angles_rad_extended.append(metadatas_clean["presented_angles_rad"].iloc[i][element]%np.pi)
            rotation_radians_extended.append(metadatas_clean["stimulus_rotations"].iloc[i][element]%np.pi)
            rotation_degrees_extended.append(metadatas_clean["stimulus_rotations_degrees"].iloc[i][element])
            expected_angles_extended.append(metadatas_clean["expected_angles"].iloc[i][element])   
            expected_angles_rad_extended.append(metadatas_clean["expected_angles_rad"].iloc[i][element]%np.pi)  

            count +=1 
    

    
    metadata_to_csv = pd.DataFrame()
    metadata_to_csv["stim_color"] = colors_extended
    metadata_to_csv["presented_angles_rad"] = presented_angles_rad_extended
    metadata_to_csv["block"] = block_extended
    metadata_to_csv["rotation_in_radian"] = rotation_radians_extended
    metadata_to_csv["rotation_in_degrees"] = rotation_degrees_extended
    metadata_to_csv["expected_angles"] = expected_angles_extended
    metadata_to_csv["expected_angles_rad"] = expected_angles_rad_extended
    metadata_to_csv["presented_angles"] = presented_angles_extended
    
    metadata_to_csv.to_csv(f'/Volumes/My Passport/Data comportement/data clean behavior MEG/metadata_for_epochs/FRINT_MEG_S{subject}_cleaned_metadata.csv', index=False)
    print(f"subject {subject} done !")
    return metadata_to_csv 

# **Metadata collection**

In [44]:
file = metadata_file_creation("12")
file 

printing subject  12 ...
subject 12 done !


,stim_color,presented_angles_rad,block,rotation_in_radian,rotation_in_degrees,fitted_value_deg,expected_angles,expected_angles_rad,presented_angles,fitted_expected_angles,fitted_expected_angles_rad,sequence_position,sequence_length
0,green,2.048682,1,1.017703,45.0,47.832522,162.380824,2.834080,117.380824,165.213346,2.883517,1,1
1,blue,2.901565,1,0.000000,0.0,0.000000,166.247420,2.901565,166.247420,166.247420,2.901565,1,5
2,blue,2.337648,1,0.000000,0.0,0.000000,133.937350,2.337648,133.937350,133.937350,2.337648,2,5
3,red,0.787955,1,2.123890,-45.0,-50.317391,180.146503,0.002557,225.146503,174.829112,3.051344,3,5
4,green,2.468017,1,1.017703,45.0,47.832522,186.406936,0.111822,141.406936,189.239458,3.302852,4,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,red,1.458072,10,2.123890,-45.0,-50.317391,38.541391,0.672674,83.541391,33.224000,0.579868,3,7
1436,red,2.107965,10,2.123890,-45.0,-50.317391,75.777505,1.322567,120.777505,70.460114,1.229761,4,7
1437,red,2.259409,10,2.123890,-45.0,-50.317391,84.454628,1.474011,129.454628,79.137237,1.381205,5,7
1438,green,0.795207,10,1.017703,45.0,47.832522,90.561993,1.580605,45.561993,93.394515,1.630042,6,7


In [45]:
corrected_angles = np.array(list_presented_angles) + np.array(rotations_to_apply)
    
    if len(corrected_angles) == 0:
        result = np.nan
    else:
        result = corrected_angles
    resulting_vector = 0
    for i in range(len(corrected_angles)):
        if not  np.isnan(result[i]):
            resulting_vector+= cmath.exp(complex(0,2* corrected_angles[i]))
        # print(np.angle(resulting_vector)/2)
    target = np.angle(resulting_vector)/2   

IndentationError: unexpected indent (4251092162.py, line 3)

In [46]:
for subject in SUBJS : 
    file = metadata_file_creation(subject)
    file

printing subject  07 ...
subject 07 done !
printing subject  09 ...
subject 09 done !
printing subject  10 ...
subject 10 done !
printing subject  11 ...
subject 11 done !
printing subject  12 ...
subject 12 done !
printing subject  14 ...
subject 14 done !


In [ ]:
for subject in SUBJS_NOT_GOOD : 
    file = metadata_file_creation_notgood(subject)
    file

printing subject  01 ...
subject 01 done !
printing subject  02 ...
subject 02 done !
printing subject  03 ...
subject 03 done !
printing subject  05 ...
subject 05 done !
printing subject  06 ...
subject 06 done !
printing subject  08 ...
subject 08 done !
